In [1]:
from langchain_core.prompts import (
                                        SystemMessagePromptTemplate,
                                        HumanMessagePromptTemplate,
                                        ChatPromptTemplate
                                        )

from langchain_core.runnables import RunnableLambda

In [2]:
# Load environment variables and configure the Groq chat model.
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv


env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file


llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)


# LangChain Expression Language Basics

LangChain Expression Language (LCEL) allows two or more runnables to be chained together into a sequence.

- The output of one runnable's `.invoke()` call becomes the input for the next runnable.
- You can create these chains using the pipe operator `|` or the `.pipe()` method.

## Types of LCEL Chains

- Sequential Chain
- Parallel Chain
- Router Chain
- Chain Runnables
- Custom Chain (Runnable Sequence)

## Sequential LCEL Chain

A Sequential LCEL Chain runs components in order, where the output from one step becomes the input for the next step.

In [3]:
system = SystemMessagePromptTemplate.from_template('You are {level} level bot . You answer in short sentences.')

question = HumanMessagePromptTemplate.from_template('tell me about the {topics} in {points} points')

messages = [system, question]
template = ChatPromptTemplate.from_messages(messages)

# Sequential Chain Here
chain = template | llm

chain.invoke({"level": "beginner", "topics": "Math", "points": 5})

AIMessage(content='1. Math uses numbers, symbols, and logic to solve problems.  \n2. It includes basic operations like addition, subtraction, multiplication, and division.  \n3. Algebra helps solve equations with unknown values.  \n4. Geometry studies shapes, sizes, and properties of space.  \n5. Calculus and statistics analyze change and data patterns.', additional_kwargs={'reasoning_content': "Okay, the user asked for a short explanation of math in five points. Let me start by breaking down the key areas. First, numbers and operations are fundamental, like addition and multiplication. Then there's algebra, which deals with variables and equations. Geometry is next, involving shapes and spatial relationships. Calculus is important for understanding change and motion. Lastly, statistics and probability help with data analysis and uncertainty. I need to keep each point concise and avoid technical jargon since the user is a beginner. Let me check if these five areas cover the main branch

In [4]:
system = SystemMessagePromptTemplate.from_template('You are {level} level bot . You answer in short sentences.')

question = HumanMessagePromptTemplate.from_template('tell me about the {topics} in {points} points')

messages = [system, question]
template = ChatPromptTemplate.from_messages(messages)
# template.invoke({"level": "beginner", "topics": "Math", "points": 5})

# Sequential Chain Here
chain = template | llm

chain.invoke({"level": "beginner", "topics": "Math", "points": 5})

AIMessage(content='1. Math is the study of numbers, shapes, and patterns.  \n2. It helps solve problems logically and analyze data.  \n3. Key areas include algebra, geometry, calculus, and statistics.  \n4. Used in science, engineering, finance, and everyday decisions.  \n5. Builds critical thinking and precision in reasoning.', additional_kwargs={'reasoning_content': "Okay, the user wants to know about math in five points. Let me start by breaking down the key areas. First, math is foundational in many fields like science and engineering. That's a good point. Next, it helps with problem-solving and logical thinking. That's another important aspect. Then, there are different branches like algebra, geometry, calculus. Maybe mention those. Also, math is used in everyday life, like managing money or measuring things. Lastly, it's a universal language that everyone uses, regardless of culture. Let me check if these points cover the essentials without being too technical. Yeah, that should 

## Parallel Chain

A Parallel Chain runs multiple runnables at the same time and combines their outputs into a single result.

In [5]:
from langchain_core.runnables import RunnableParallel

In [6]:
template = ChatPromptTemplate.from_template("Give me a Poem in Cricket")

# Sequential Chain Here
chain1 = template | llm

In [7]:
template = ChatPromptTemplate.from_template("Give me a Poem in football")

# Sequential Chain Here
chain2 = template | llm

In [8]:
parallel_chain = RunnableParallel(
    cricket=chain1,
    football=chain2,
)
parallel_chain.invoke({})

{'cricket': AIMessage(content="**Ode to the Cricket's Dance**  \n\nOn emerald fields where legends play,  \nWhere echoes of the past hold sway,  \nThe bat, a willow’s whispered might,  \nMeets ball in a symphony of flight.  \n\nThe bowler leans, a coiled refrain,  \nHis fingers grip the game’s tight chain.  \nThe batsman stands, a sentinel,  \nTime pauses—then, the clash is keen.  \n\nThe pitch, a stage for fate’s design,  \nWhere hopes in stitches intertwine.  \nA spinner’s guile, a seamer’s hiss—  \nThe air grows thick with tension’s kiss.  \n\nThe crowd, a tide of gasps and cheers,  \nBreathes life into the silent spheres.  \nA six, a roar! A catch, a sigh—  \nThe scoreboard weeps and wonders why.  \n\nVictory wears a thousand grins,  \nWhile defeat, in shadows, etches sins.  \nYet both, in cricket’s code, are bound,  \nBy the grace of the ground, the fall, the crown.  \n\nFor this is more than sticks and score—  \nIt’s dawn’s first light, the final core.  \nA dance of time, from du

## Router Chain

A Router Chain directs the input to different runnable paths based on the input type or condition.

In [9]:
template = ChatPromptTemplate.from_template("Give a Happy Message to user")

# Sequential Chain Here
chain1 = template | llm

chain1.invoke({})

AIMessage(content="🌈 Hello there! Wishing you a day filled with little joys, big smiles, and moments that make your heart skip a beat! 🌟 Whether it's the sun shining just right, a cozy cup of tea, or a laugh with someone you love—cherish it all. You’re deserving of every happy thing life has to offer. Keep spreading positivity, and remember: good vibes are contagious! 💫✨ Have an absolutely wonderful day! 🌸", additional_kwargs={'reasoning_content': 'Okay, the user wants a happy message. Let me think about what makes a message truly uplifting. Maybe start with a friendly greeting. Use emojis to add some warmth. Include something about spreading joy and positivity. Maybe mention little things that bring happiness. Add wishes for good vibes and a bright day. Keep it short and cheerful. Let me put that all together in a natural way.\n'}, response_metadata={'token_usage': {'completion_tokens': 176, 'prompt_tokens': 14, 'total_tokens': 190, 'completion_time': 0.432717738, 'completion_tokens_d

In [10]:
template = ChatPromptTemplate.from_template("Give me a Sad Message to the user")

# Sequential Chain Here
chain2 = template | llm

chain2.invoke({})

AIMessage(content="Here’s a message crafted to convey quiet sorrow:  \n\n---  \n\nThe weight of some days isn’t in the tears we cry, but in the ones we swallow, letting them pool in our throats until even silence feels like a stranger. You taught me how to miss someone so deeply it aches in my bones, how to hold a memory like a ghost—familiar, but always just out of reach. I wonder if the stars still remember your name, or if they’ve forgotten how to shine for things that no longer are.  \n\n---  \n\nLet me know if you'd like adjustments—this can be tweaked to fit a specific tone or context.", additional_kwargs={'reasoning_content': 'Okay, the user wants a sad message. Let me think about how to approach this. They might be looking for something heartfelt, maybe to express their own feelings or to share with someone else. I need to make sure the message is genuine and not too over the top. Let me start by considering common themes of sadness—loneliness, loss, longing.\n\nI should use si

In [11]:
def route(info):
    if "happy" in info["sentiment"]:
        return chain1.invoke({})
    elif "sad" in info["sentiment"]:
        return chain2.invoke({})

In [12]:
def return_sentiment(info):
    template = ChatPromptTemplate.from_template("Based on the User statement {statement} either give a single word output: happy or sad")
    chain = template | llm
    sentiment = chain.invoke({"statement": info["statement"]}).content.strip().lower()
    return {"sentiment" : sentiment}

In [23]:
statement = "I am feeling great today!"

result_sentiment = return_sentiment({"statement": statement})
full_chain = {"sentiment": RunnableLambda(lambda x: result_sentiment["sentiment"])} | RunnableLambda(route)
full_chain.invoke({})

AIMessage(content="🌈 Hello there! 🌟 Wishing you a day filled with sunshine, smiles, and little moments that make your heart skip a beat! 🌸 Whether it's a kind word, a cozy cup of tea, or a laugh with loved ones—cherish it all. You’re capable of amazing things, and the world is brighter because you’re in it. Keep shining, and never forget how wonderful you are! ✨💖 #HappyVibesOnly 🌟", additional_kwargs={'reasoning_content': 'Okay, the user wants a happy message. Let me think about what makes a message truly uplifting. Maybe start with a cheerful greeting. Use emojis to add some visual positivity. Include something about spreading joy and kindness. Maybe mention little things that bring happiness. Add a motivational note about each day being a new opportunity. Keep it short and warm. Let me put it all together in a friendly tone.\n'}, response_metadata={'token_usage': {'completion_tokens': 183, 'prompt_tokens': 14, 'total_tokens': 197, 'completion_time': 0.448554792, 'completion_tokens_de

In [22]:
statement = "I am feeling great today!"

def return_sentiment_1(info):
    template = ChatPromptTemplate.from_template("Based on the User statement {statement} either give a single word output: happy or sad")
    chain = template | llm
    sentiment = chain.invoke({"statement": info["statement"]}).content.strip().lower()
    return {"sentiment": sentiment}

def route_1(info):
    if "happy" in info["sentiment"]:
        return chain1
    elif "sad" in info["sentiment"]:
        return chain2

full_chain = {"statement": RunnableLambda(lambda x: statement)} | RunnableLambda(return_sentiment) | RunnableLambda(route_1)
full_chain.invoke({})

AIMessage(content="🌈 Hello there! Wishing you a day filled with little joys, big smiles, and moments that make your heart skip a beat! 🌸 Whether it's the warmth of the sun, a kind word from a friend, or your favorite song playing at just the right time—cherish it all. You’re deserving of happiness, so let’s spread positivity like confetti! 🎉✨ Have a splendid day! 😊", additional_kwargs={'reasoning_content': 'Okay, the user wants a happy message. Let me think about what makes a message truly uplifting. Maybe start with a cheerful greeting. Use some emojis to add a friendly vibe. Include something about spreading joy and positivity. Maybe mention small joys in life. Add a metaphor like sunshine or flowers. Encourage the person to smile and take a moment to appreciate good things. Keep it short and warm. Let me put that all together in a natural way.\n'}, response_metadata={'token_usage': {'completion_tokens': 184, 'prompt_tokens': 14, 'total_tokens': 198, 'completion_time': 0.484887192, '

## Chain Runnables

Chain Runnables are reusable runnable components that can be combined to build larger LCEL workflows.

- `RunnableLambda` runs a normal Python function as a runnable.
- `RunnablePassthrough` passes the input forward without changing it.

In [ ]:
from langchain_core.runnables import RunnableLambda

system = SystemMessagePromptTemplate.from_template(
    "You are a {level} level bot. You answer in short sentences."
)
question = HumanMessagePromptTemplate.from_template(
    "Tell me about {topics} in {points} points."
)

messages = [system, question]
template = ChatPromptTemplate.from_messages(messages)

chain1 = template | llm
result =  chain1.invoke({"level": "beginner", "topics": "Math", "points": 5})

In [ ]:
def follow_up_template_function(message : dict):
    follow_up_template = ChatPromptTemplate.from_template(
    f"Summarize the following answer in 3 short bullet points:\n\n{message['check']}"
    )
    
    return follow_up_template

chain2 = (
    RunnableLambda(follow_up_template_function)
    | llm
)

chain2.invoke({"check": result.content})

AIMessage(content='- Mathematics explores numbers, patterns, and relationships through branches like algebra, geometry, and calculus.  \n- It applies to real-world problem-solving in areas such as budgeting, engineering, and data analysis.  \n- Develops logical reasoning and analytical skills essential for science, technology, and advanced studies.', additional_kwargs={'reasoning_content': "Okay, let's see. The user wants me to summarize the given answer into three short bullet points. The original answer has five points. I need to condense them without losing the key information.\n\nFirst, the main aspects of math: it's about numbers, patterns, relationships. Then, real-world applications like budgeting and engineering. The branches are algebra, geometry, etc. Also, it builds logical thinking and is foundational for other fields.\n\nHmm, how to group these? Maybe combine the first point with the branches. Then the applications and skills. Finally, the foundational role. Let me check i

In [ ]:
# RunnablePassthrough

from langchain_core.runnables import RunnablePassthrough, RunnableLambda

## Custom Chain (Runnable Sequence)

A Custom Chain uses a runnable sequence to connect multiple custom steps into a flexible workflow.